# Minimal RLM Spark log root-cause analysis

Create a synthetic Spark log, find the failed job and executor, and check the answer against known facts. No Spark job or uploaded log is needed.

Use Microsoft Fabric with a default Lakehouse attached and write access to `/lakehouse/default/Files/fabric_rlm_examples`. Select Python 3.12 in the Fabric UI where available. The model call uses your notebook's Fabric identity and requires access to Fabric AI services. Confirm model availability for your workspace and region using the [model list](https://learn.microsoft.com/en-us/fabric/data-science/ai-services/ai-services-overview#consumption-rate); the configured model and reasoning settings are examples, not guarantees.

Run the pinned install, **restart the Python session**, then run the remaining cells, not the install cell. Let pip resolve dependencies; see [Fabric environment setup](../../docs/fabric-runtime-deps.md) for troubleshooting. There are no local fallbacks or suppressed exceptions. Offline tests validate this fixture and scripted runtime submissions, not full Fabric execution or live model quality.

In [ ]:
%pip install -q fabric-rlm==0.6.7

## Write the fixture

Each run overwrites this example file. Job 16 succeeds on executor 3; job 17 fails in stage 42 after executor 7 runs out of Java heap space. The shutdown message is not the root cause.

In [ ]:
from pathlib import Path

source_dir = Path("/lakehouse/default/Files/fabric_rlm_examples") / "spark_log" / "sources"
source_dir.mkdir(parents=True, exist_ok=True)
log_path = source_dir / "spark_app.log"
log_path.write_text(
    """
24/01/01 10:00:00 INFO DAGScheduler: Job 16 finished in stage 41.0 on executor 3
24/01/01 10:00:01 INFO DAGScheduler: Got job 17 collect at SparkApp.scala:142
24/01/01 10:01:30 WARN TaskSetManager: Lost task 88001.0 in stage 42.0 on worker-07.fabric.local executor 7: java.lang.OutOfMemoryError
24/01/01 10:02:31 WARN TaskSetManager: Lost task 88001.1 in stage 42.0 on worker-07.fabric.local executor 7: java.lang.OutOfMemoryError
24/01/01 10:03:32 WARN TaskSetManager: Lost task 88001.2 in stage 42.0 on worker-07.fabric.local executor 7: java.lang.OutOfMemoryError
24/01/01 10:04:33 WARN TaskSetManager: Lost task 88001.3 in stage 42.0 on worker-07.fabric.local executor 7: java.lang.OutOfMemoryError
24/01/01 10:04:33 ERROR TaskSetManager: Task 88001 in stage 42.0 failed 4 times; aborting job
24/01/01 10:04:34 ERROR DAGScheduler: Job 17 failed: collect at SparkApp.scala:142, took 273.0 s
24/01/01 10:04:35 ERROR YarnScheduler: Lost executor 7 on worker-07.fabric.local: java.lang.OutOfMemoryError: Java heap space
24/01/01 10:04:36 INFO SparkContext: Successfully stopped SparkContext
    """.strip(),
    encoding="utf-8",
)
print(log_path)
print(log_path.read_text(encoding="utf-8"))

## Analyze

`File` gives the worker a path, not an automatic prompt attachment. Generated code can read and print data back to the model; it is not a privacy boundary. This tiny example needs only standard-library text parsing. Typed outputs enforce Python types, not factual correctness. The `.run()` below makes model calls and may incur capacity usage.

In [ ]:
from fabric_rlm import FabricLM, File, RLM

rlm = RLM.task(
    task=(
        "Analyze the Spark log using Python standard-library text parsing. "
        "Identify the failed job, failed stage, and failing executor as integer IDs. "
        "For root_cause, return the exact exception class and message from the log, "
        "without the timestamp or logger prefix. For evidence, return a list of "
        "verbatim log lines supporting the job failure, stage failure, and root cause."
    ),
    inputs={"log_file": File(log_path)},
    outputs={
        "failed_job_id": int, "failed_stage_id": int, "failing_executor": int,
        "root_cause": str, "evidence": list,
    },
    lm=FabricLM("gpt-5.1", reasoning_effort="low"),
    max_turns=6,
)

result = rlm.run()

## Check the answer

These checks are independent of the model and apply only to this synthetic fixture. They fail visibly on an incorrect answer. A real incident needs its own evidence review; heap exhaustion alone does not establish why memory demand was excessive.

In [ ]:
assert result.submitted, result.report()
expected = {
    "failed_job_id": 17, "failed_stage_id": 42, "failing_executor": 7,
}
assert {key: result.outputs[key] for key in expected} == expected, result.outputs
# A model may quote the log line around the exception; the exception itself must be in it.
ROOT_CAUSE = "java.lang.OutOfMemoryError: Java heap space"
assert ROOT_CAUSE in result.outputs["root_cause"], result.outputs
evidence = result.outputs["evidence"]
log_lines = log_path.read_text(encoding="utf-8").splitlines()
assert evidence and all(isinstance(line, str) and line in log_lines for line in evidence), evidence
for marker in ("Job 17 failed", "stage 42.0 failed", ROOT_CAUSE):
    assert any(marker in line for line in evidence), (marker, evidence)
print(result.report())
print("Root cause:", result.outputs["root_cause"])
print("Evidence:\n" + "\n".join(evidence))